# LC 297 — Serialize and Deserialize Binary Tree

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> BFS level-order with explicit "null"
markers lets you encode the tree's shape completely — deserialization
simply replays the BFS, re-attaching children in the same order
they were recorded.
</div>

## Official Problem Statement

Serialization is the process of converting a data structure or object
into a sequence of bits so that it can be stored in a file or memory
buffer, or transmitted across a network connection link to be
reconstructed later in the same or another computer environment.

Design an algorithm to serialize and deserialize a binary tree. There
is no restriction on how your serialization/deserialization algorithm
should work. You just need to ensure that a binary tree can be
serialized to a string and this string can be deserialized to the
original tree structure.

**Constraints:**
- The number of nodes in the tree is in the range `[0, 10^4]`
- `-1000 <= Node.val <= 1000`

## What This Is Actually Asking

You need to flatten a tree into a string and reconstruct it exactly.
The hard part is encoding missing children so the shape is preserved
— without null markers, you can't tell a leaf from a node with one
child. BFS level-order naturally produces the encoding because
you process parents before children, so reconstruction can follow
the same queue-based order.

## Walk Through an Example by Hand

```
Tree:   1
       / \
      2   3
         / \
        4   5

Serialize (BFS):
  Queue: [1]
  Pop 1  -> output "1,", push 2, push 3
  Pop 2  -> output "2,", push null, null
  Pop 3  -> output "3,", push 4, push 5
  Pop null -> output "null,"
  Pop null -> output "null,"
  Pop 4  -> output "4,", push null, null
  Pop 5  -> output "5,"
  Result: "1,2,3,null,null,4,5"

Deserialize:
  tokens = ["1","2","3","null","null","4","5"]
  root=1, queue=[1]
  Pop 1: left=2, right=3, queue=[2,3]
  Pop 2: left=null, right=null, queue=[3]
  Pop 3: left=4, right=5, queue=[4,5]
  Pop 4: left=null, right=null
  Pop 5: left=null, right=null
```

## The Picture

```
SERIALIZE (BFS wave by wave):

Level 0:        [1]           -> "1,"
Level 1:      [2] [3]         -> "2,3,"
Level 2:  [N][N][4][5]        -> "null,null,4,5"

Encoded string: "1,2,3,null,null,4,5"
                 ↑       ↑↑↑↑↑↑
                root     shape markers

DESERIALIZE (mirror BFS):

 tokens:  1   2   3  null null  4   5
          ↑   ↑   ↑   ↑    ↑   ↑   ↑
 token[0] = root
 Queue processes each node, consuming next 2 tokens
 as left child, right child:

 node=1 -> left=tokens[1]=2, right=tokens[2]=3
 node=2 -> left=tokens[3]=null, right=tokens[4]=null
 node=3 -> left=tokens[5]=4, right=tokens[6]=5
```

## When To Use This Pattern

- When you need to **persist or transmit a tree**, think
  **BFS serialization with null markers**.
- When the problem says "design an encoder/decoder", think
  **level-order with a reconstruction queue**.
- When you need **round-trip fidelity** (serialize then
  deserialize returns the exact structure), think
  **explicit null tokens**.
- When DFS preorder is simpler to code, think **sentinel '#'
  for null** as an alternative.
- When storing tree configs in JSON/DB, think **this pattern
  directly maps to array representation**.

## The Approach

Serialize with BFS: process each node, emit its value, push its
children (or "null") to the queue, and join tokens with commas.
Deserialize by splitting on commas, making the first token the
root, then using a queue of parent nodes — each parent consumes
the next two tokens as its left and right children, queueing
non-null children for their own turn.

In [ ]:
from collections import deque
from typing import Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def make_tree(vals):
    if not vals:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """BFS to list for comparison."""
    if not root:
        return []
    result, q = [], deque([root])
    while q:
        node = q.popleft()
        if node:
            result.append(node.val)
            q.append(node.left)
            q.append(node.right)
        else:
            result.append(None)
    # trim trailing Nones
    while result and result[-1] is None:
        result.pop()
    return result

In [ ]:
def test_harness(codec_class):
    cases = [
        [1, 2, 3, None, None, 4, 5],
        [],
        [1],
        [1, 2],
        [1, None, 2, None, 3],
    ]
    passed = 0
    codec = codec_class()
    for i, vals in enumerate(cases):
        root = make_tree(vals)
        data = codec.serialize(root)
        recovered = codec.deserialize(data)
        original_l = tree_to_list(root)
        recovered_l = tree_to_list(recovered)
        ok = original_l == recovered_l
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(
                f"  Case {i}: orig={original_l} "
                f"recovered={recovered_l}"
            )
        print(f"  Case {i}: {status}  encoded='{data}'")
    print(f"\nSummary: {passed}/{len(cases)} passed")

In [ ]:
class Codec:
    """
    Serialize and deserialize a binary tree via BFS.

    serialize: BFS level-order, null for missing nodes,
               tokens joined by comma.
    deserialize: split tokens, BFS reconstruction —
                 each queued node consumes two tokens.
    """

    def serialize(self, root: Optional[TreeNode]) -> str:
        """
        Encode tree to a single string.
        """
        if not root:
            print("[DEBUG] serialize: empty tree")
            pass  # return ""
        tokens = []
        q = deque([root])
        while q:
            node = q.popleft()
            if node:
                tokens.append(str(node.val))
                q.append(node.left)
                q.append(node.right)
                print(f"[DEBUG] serialize: emitting {node.val}")
            else:
                tokens.append("null")
        pass  # return ",".join(tokens)

    def deserialize(self, data: str) -> Optional[TreeNode]:
        """
        Decode string back to binary tree.
        """
        if not data:
            pass  # return None
        tokens = deque(data.split(","))
        root_val = tokens.popleft()
        if root_val == "null":
            pass  # return None
        root = TreeNode(int(root_val))
        q = deque([root])
        while q:
            node = q.popleft()
            left_val = tokens.popleft()
            if left_val != "null":
                node.left = TreeNode(int(left_val))
                q.append(node.left)
                print(
                    f"[DEBUG] deserialize: "
                    f"{node.val}.left = {left_val}"
                )
            right_val = tokens.popleft()
            if right_val != "null":
                node.right = TreeNode(int(right_val))
                q.append(node.right)
                print(
                    f"[DEBUG] deserialize: "
                    f"{node.val}.right = {right_val}"
                )
        pass  # return root

In [ ]:
# Uncomment and run when solution is ready
# test_harness(Codec)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force (no null marks, preorder only) | O(N) — but lossy | O(N) |
| **BFS with null markers (optimal)** | **O(N)** | **O(N)** |
| DFS preorder with sentinel '#' | O(N) | O(N) |

N = number of nodes. The encoded string length is O(N) in all
non-brute approaches. BFS queue holds at most O(W) nodes where
W is the max width.

## Real World Connection

Financial products like option strategies have tree-like payoff
structures that must be serialized to JSON for storage in Citi's
pricing databases. AWS DynamoDB stores hierarchical configs as
serialized trees in string attributes. In data engineering,
Apache Spark's query plan (a tree of operators) is serialized
for transfer between driver and executors. XML and JSON
themselves are tree serialization formats — this problem is
essentially re-inventing them from scratch.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra